# Step 1: Load the data

This notebook loads every file for the Student Rent Finder project and checks it looks right.

**How to run it**
1. Top-right corner: **Select Kernel** → **Python Environments** → choose your `.venv` (or Anaconda `base`).
2. Click **Run All** at the top, or press **Shift + Enter** on each cell in order.

Each section ends with a ✅ line telling you what to expect.

## 1. Where the data lives

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)

DATA = Path(r"C:\Users\User\Desktop\Sanuth\data")   # your data folder
PROC = DATA / "processed"                              # cleaned files

for p in sorted(DATA.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(DATA)!s:<75} {p.stat().st_size / 1e6:8.1f} MB")

✅ You should see the rent, timetable (`gtfs.zip`), street map and suburb boundary files in `data`, the two notebooks, and 7 files in `processed`.

## 2. The raw rent file (as downloaded)

One sheet per property type. Row 1 has the quarters, row 2 says Count / Median, suburbs start on row 3.

In [ ]:
RENT_XLSX = next(DATA.glob("Moving annual median rent*.xlsx"))
xl = pd.ExcelFile(RENT_XLSX)
print(xl.sheet_names)
pd.read_excel(RENT_XLSX, sheet_name="1 bedroom flat", header=None).iloc[:6, :8]

✅ 7 sheet names, and a table whose first suburb is *Albert Park-Middle Park-West St Kilda*.

## 3. The cleaned rent data

- `rents_long`: every quarter since 2000 (for trends and forecasting later)
- `rents_latest`: the latest quarter only, areas with at least 10 new leases (for the app)

In [ ]:
rents_long = pd.read_csv(PROC / "rents_long.csv", parse_dates=["quarter_date"])
rents_latest = pd.read_csv(PROC / "rents_latest.csv", parse_dates=["quarter_date"])

print("rents_long:  ", rents_long.shape, "|", rents_long.quarter_date.min().date(), "to", rents_long.quarter_date.max().date())
print("rents_latest:", rents_latest.shape)
rents_long.head()

In [ ]:
# Typical weekly rent by property type, latest quarter
rents_latest.groupby("property_type")["median"].describe()[["count", "min", "50%", "max"]].round(0)

✅ About 101,000 rows from March 2000 to September 2025; a 1-bedroom flat's median around $400 a week.

## 4. How rents have changed (quick chart)

In [ ]:
import matplotlib.pyplot as plt

areas = ["Carlton-Parkville", "Clayton", "Footscray", "Box Hill", "Brunswick"]
one_bed = rents_long[(rents_long.property_type == "1 bedroom flat") & rents_long.suburb_group.isin(areas)]

fig, ax = plt.subplots(figsize=(10, 5))
for name, g in one_bed.groupby("suburb_group"):
    ax.plot(g.quarter_date, g["median"], label=name)
ax.set_title("Median weekly rent, 1-bedroom flat (moving annual)")
ax.set_ylabel("$ per week")
ax.legend()
plt.show()

✅ Lines rising over time, with a jump after 2022. (A name with no line isn't in the rental report under that exact name; try another from `rents_long.suburb_group.unique()`.)

## 5. Suburbs, campuses and the lookup table

- `group_lookup`: which ABS suburbs belong to each rental area
- `rents_by_suburb`: latest rent for each suburb and property type
- `suburb_points`: the centre point of each Melbourne suburb (used for travel times)
- `campuses`: the 7 university campuses

In [ ]:
lookup = pd.read_csv(PROC / "group_lookup.csv", dtype={"SAL_CODE21": str})
rents_by_suburb = pd.read_csv(PROC / "rents_by_suburb.csv", dtype={"SAL_CODE21": str})
suburb_points = pd.read_csv(PROC / "suburb_points.csv", dtype={"id": str})
campuses = pd.read_csv(PROC / "campuses.csv")

print("rental areas:", lookup.suburb_group.nunique(), "| suburbs linked:", len(lookup),
      "| in Melbourne map:", int(lookup.in_melbourne_map.sum()))
print("rents_by_suburb:", rents_by_suburb.shape, "| suburb_points:", suburb_points.shape)
campuses

✅ 145 rental areas, 213 suburbs, 163 in Melbourne, and 7 campuses.

## 6. Map: suburbs coloured by 1-bedroom rent, with campuses

In [ ]:
import json
import folium

shapes = json.loads((PROC / "suburb_shapes.geojson").read_text())
rent_1b = (rents_by_suburb[rents_by_suburb.property_type == "1 bedroom flat"]
           .set_index("SAL_CODE21")["median"].to_dict())

def colour(rent):
    if rent is None: return "#cccccc"
    if rent < 350:  return "#1a9850"
    if rent < 425:  return "#91cf60"
    if rent < 500:  return "#fee08b"
    if rent < 575:  return "#fc8d59"
    return "#d73027"

for f in shapes["features"]:
    f["properties"]["rent"] = rent_1b.get(f["properties"]["SAL_CODE21"])

m = folium.Map(location=[-37.81, 144.96], zoom_start=10, tiles="CartoDB positron")
folium.GeoJson(
    shapes,
    style_function=lambda f: {"fillColor": colour(f["properties"]["rent"]),
                              "color": "#555", "weight": 0.5, "fillOpacity": 0.7},
    tooltip=folium.GeoJsonTooltip(["suburb", "suburb_group", "rent"],
                                  aliases=["Suburb", "Rental area", "1-bed median $/wk"]),
).add_to(m)
for _, c in campuses.iterrows():
    folium.Marker([c.lat, c.lon], tooltip=c.campus, icon=folium.Icon(color="blue", icon="graduation-cap", prefix="fa")).add_to(m)
m

✅ An interactive map of Melbourne: green is cheaper, red is dearer, blue pins are campuses. Hover over a suburb to see its rent.

## 7. Timetables and street map

These are used in the next step (travel times). This checks they're readable and the timetable dates.

In [ ]:
import zipfile, io

with zipfile.ZipFile(DATA / "gtfs.zip") as z:
    feeds = sorted(n for n in z.namelist() if n.endswith("google_transit.zip"))
    with zipfile.ZipFile(io.BytesIO(z.read("2/google_transit.zip"))) as trains:
        cal = pd.read_csv(trains.open("calendar.txt"), dtype=str)

print("timetable feeds:", feeds)
print("metro train timetable valid:", cal.start_date.min(), "to", cal.end_date.max())
print("street map:", (DATA / "Melbourne.osm.pbf").stat().st_size / 1e6, "MB")

✅ 8 timetable feeds (2 = metro train, 3 = tram, 4 = metro bus), valid 20261030 to 20261231, and a street map of about 90 MB.

## Done

All the data is loaded. Next is **Step 3: travel times** — open `03_travel_times.ipynb` in this folder.